# SDG Classifier — Embedding flow demo

This notebook demonstrates: loading an embedding model, instantiating `SDGClassifier` with embeddings, automatically saving/loading the computed phrase and FAISS indices with a cache directory, and running an `embed_ngram` classification.

## Setup
Install the project dependencies from `requirements.txt`. The requirements file includes `sentence-transformers` for embedding methods and `faiss-cpu` for FAISS acceleration.
```bash
pip install -r requirements.txt
```
The notebook will use a local `cache/` folder to store `phrase_index.pkl` and FAISS index files automatically.

## Similarity methods

This demo uses three related matching modes:

- `similarity_threshold`: a word-level fallback for small spelling or pluralization differences.
- `similarity_method='embed'`: whole-document bi-encoder matching.
- `similarity_method='embed_ngram'`: sliding n-gram bi-encoder matching for short phrase detection inside longer text.

The later code cells show how each option changes the output on the same example text.


In [ ]:
# Automatic cache persistence demo
# This builds a classifier with FAISS enabled and cache paths configured so phrase and index artifacts
# are saved automatically and can be restored on the next run.

import importlib
import os
from pathlib import Path
import sys


def _bootstrap_project_root():
    here = Path.cwd().resolve()
    candidates = [here, here.parent, here.parent.parent]
    for base in candidates:
        if (base / 'sdg_classifier.py').exists():
            root = base
            break
        if (base / 'ElsevierMapping' / 'sdg_classifier.py').exists():
            root = base / 'ElsevierMapping'
            break
    else:
        root = None
    if root is not None and str(root) not in sys.path:
        sys.path.insert(0, str(root))
    return root


project_root = _bootstrap_project_root()
if project_root is None:
    raise RuntimeError('Could not locate sdg_classifier.py from the current notebook working directory.')

sdg_dir = project_root / 'SDG Queries'
print('Using project root:', project_root)
print('Using SDG query directory:', sdg_dir)

import sdg_classifier as sdg_module
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')


def embedder(texts):
    arr = embed_model.encode(texts, convert_to_numpy=False)
    return [list(a) for a in arr]


sdg_module = importlib.reload(sdg_module)
SDGClassifier = sdg_module.SDGClassifier

cache_dir = Path('cache')
cache_faiss_dir = cache_dir / 'faiss'

cached_classifier = SDGClassifier(
    sdg_dir_path=str(sdg_dir) if sdg_dir.exists() else None,
    embedder=embedder,
    use_faiss=True,
    cache_dir=str(cache_dir),
    cache_faiss_dir=str(cache_faiss_dir),
    auto_load_cache=True,
    auto_save_cache=True,
)

print('Loaded phrase index groups:', sum(len(v) for v in cached_classifier.phrase_index.values()))
print('Loaded FAISS groups:', sum(len(v) for v in cached_classifier.faiss_index.values()) if cached_classifier.faiss_index else 0)
print('Saved phrase index exists:', (cache_dir / 'phrase_index.pkl').exists())
print('Saved FAISS files:', [name for name in os.listdir(cache_faiss_dir) if name.endswith('.index')] if cache_faiss_dir.is_dir() else [])

cached_vec = cached_classifier.classify(
    'Reducing poverty and hunger in rural communities',
    'We investigate social protection and safety nets to tackle food insecurity and extreme poverty in low income regions.',
    ['poverty', 'social protection'],
    similarity_threshold=0.75,
    similarity_method='embed_ngram',
)
print('embed_ngram vector from cached classifier:', cached_vec)

Using project root: C:\Users\tziou\OneDrive - Cyprus University of Technology\Research\SDGs\ElsevierMapping
Using SDG query directory: C:\Users\tziou\OneDrive - Cyprus University of Technology\Research\SDGs\ElsevierMapping\SDG Queries


c:\Users\tziou\OneDrive - Cyprus University of Technology\Research\SDGs\ElsevierMapping\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\tziou\OneDrive - Cyprus University of Technology\Research\SDGs\ElsevierMapping\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tziou\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To su

In [ ]:
# Similarity-threshold example
# This uses the word-level similarity fallback and is useful when the text is close,
# but not an exact match to the SDG phrases.

from pathlib import Path
import sys

def _bootstrap_project_root():
    here = Path.cwd().resolve()
    candidates = [here, here.parent, here.parent.parent]
    for base in candidates:
        if (base / 'sdg_classifier.py').exists():
            root = base
            break
        if (base / 'ElsevierMapping' / 'sdg_classifier.py').exists():
            root = base / 'ElsevierMapping'
            break
    else:
        root = None
    if root is not None and str(root) not in sys.path:
        sys.path.insert(0, str(root))
    return root

project_root = _bootstrap_project_root()
print('Using project root:', project_root)

from sdg_classifier import SDGClassifier

classifier = SDGClassifier()

threshold_title = 'Reducing poverty and food insecurity in rural communities'
threshold_abstract = (
    'We study social protection, safety nets, and related policies to reduce '
    'extreme poverty and hunger in low-income regions.'
)
threshold_keywords = ['poverty', 'social protection']

threshold_vec = classifier.classify(
    threshold_title,
    threshold_abstract,
    threshold_keywords,
    similarity_threshold=0.8,
)
print('SDG vector with similarity_threshold=0.8:', threshold_vec)
print('Matched SDGs:', [i + 1 for i, value in enumerate(threshold_vec) if value] or '(none)')

In [ ]:
# Exact-match vs similarity-threshold comparison
# This shows how the word-level fallback changes results compared with exact matching.

from pathlib import Path
import sys

def _bootstrap_project_root():
    here = Path.cwd().resolve()
    candidates = [here, here.parent, here.parent.parent]
    for base in candidates:
        if (base / 'sdg_classifier.py').exists():
            root = base
            break
        if (base / 'ElsevierMapping' / 'sdg_classifier.py').exists():
            root = base / 'ElsevierMapping'
            break
    else:
        root = None
    if root is not None and str(root) not in sys.path:
        sys.path.insert(0, str(root))
    return root

project_root = _bootstrap_project_root()
print('Using project root:', project_root)

from sdg_classifier import classify_sdgs

compare_title = 'Reducing poverty and food insecurity in rural communities'
compare_abstract = (
    'We study social protection, safety nets, and related policies to reduce '
    'extreme poverty and hunger in low-income regions.'
)
compare_keywords = ['poverty', 'social protection']

exact_vec = classify_sdgs(compare_title, compare_abstract, compare_keywords)
threshold_vec = classify_sdgs(
    compare_title,
    compare_abstract,
    compare_keywords,
    similarity_threshold=0.8,
)

print('Exact-match SDG vector:      ', exact_vec)
print('Thresholded SDG vector:      ', threshold_vec)
print('Exact-match SDGs:            ', [i + 1 for i, value in enumerate(exact_vec) if value] or '(none)')
print('Thresholded SDGs:            ', [i + 1 for i, value in enumerate(threshold_vec) if value] or '(none)')
print('New matches from thresholding:', [i + 1 for i, (e, t) in enumerate(zip(exact_vec, threshold_vec)) if not e and t] or '(none)')

In [ ]:
# Compare all supported similarity_method values side by side
# similarity_method=None   -> exact matching + SequenceMatcher fallback via similarity_threshold
# similarity_method='embed' -> whole-document bi-encoder
# similarity_method='embed_ngram' -> sliding n-gram bi-encoder

from pathlib import Path
import sys


def _bootstrap_project_root():
    here = Path.cwd().resolve()
    candidates = [here, here.parent, here.parent.parent]
    for base in candidates:
        if (base / 'sdg_classifier.py').exists():
            root = base
            break
        if (base / 'ElsevierMapping' / 'sdg_classifier.py').exists():
            root = base / 'ElsevierMapping'
            break
    else:
        root = None
    if root is not None and str(root) not in sys.path:
        sys.path.insert(0, str(root))
    return root


project_root = _bootstrap_project_root()
if project_root is None:
    raise RuntimeError('Could not locate sdg_classifier.py from the current notebook working directory.')

sdg_dir = project_root / 'SDG Queries'
print('Using project root:', project_root)

from sdg_classifier import SDGClassifier, classify_sdgs
from sentence_transformers import SentenceTransformer

compare_title = 'Reducing poverty and food insecurity in rural communities'
compare_abstract = (
    'We study social protection, safety nets, and related policies to reduce '
    'extreme poverty and hunger in low-income regions.'
)
compare_keywords = ['poverty', 'social protection']

# Exact / threshold baseline uses the module-level classifier.
exact_vec = classify_sdgs(compare_title, compare_abstract, compare_keywords)
threshold_vec = classify_sdgs(
    compare_title,
    compare_abstract,
    compare_keywords,
    similarity_threshold=0.9,
)

embed_model = SentenceTransformer('all-MiniLM-L6-v2')


def embedder(texts):
    arr = embed_model.encode(texts, convert_to_numpy=False)
    return [list(a) for a in arr]


compare_classifier = SDGClassifier(
    sdg_dir_path=str(sdg_dir) if sdg_dir.exists() else None,
    embedder=embedder,
    use_faiss=True,
)
embed_vec = compare_classifier.classify(
    compare_title,
    compare_abstract,
    compare_keywords,
    similarity_threshold=0.85,
    similarity_method='embed',
)
embed_ngram_vec = compare_classifier.classify(
    compare_title,
    compare_abstract,
    compare_keywords,
    similarity_threshold=0.90,
    similarity_method='embed_ngram',
)

print('Exact-match vector:          ', exact_vec)
print('Thresholded vector:          ', threshold_vec)
print('Embed vector:                ', embed_vec)
print('Embed-ngram vector:          ', embed_ngram_vec)
print('Exact-match SDGs:            ', [i + 1 for i, value in enumerate(exact_vec) if value] or '(none)')
print('Thresholded SDGs:            ', [i + 1 for i, value in enumerate(threshold_vec) if value] or '(none)')
print('Embed SDGs:                  ', [i + 1 for i, value in enumerate(embed_vec) if value] or '(none)')
print('Embed-ngram SDGs:            ', [i + 1 for i, value in enumerate(embed_ngram_vec) if value] or '(none)')
print('New matches from thresholding:', [i + 1 for i, (e, t) in enumerate(zip(exact_vec, threshold_vec)) if not e and t] or '(none)')